In [ ]:
import os, time, copy
from matplotlib import pyplot as plt
import seaborn as sns
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report, confusion_matrix)
from PIL import Image

# Device Setup

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
torch.backends.cudnn.benchmark = True

# Data Transforms

In [ ]:
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]
data_transforms = {
    "train": transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
        transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
        transforms.RandomPerspective(distortion_scale=0.3, p=0.3),
        transforms.RandomGrayscale(p=0.05),
        transforms.ToTensor(), transforms.Normalize(mean, std),
    ]),
    "val":  transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                                  transforms.ToTensor(), transforms.Normalize(mean, std)]),
    "test": transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
                                  transforms.ToTensor(), transforms.Normalize(mean, std)]),
}

# Dataset & Dataloaders

In [ ]:
DATA_DIR = "Dataset"
image_datasets = {x: datasets.ImageFolder(os.path.join(DATA_DIR, x), data_transforms[x])
                  for x in ["train", "val", "test"]}
dataloaders = {x: DataLoader(image_datasets[x], batch_size=32, shuffle=(x=="train"),
                              pin_memory=True, num_workers=6)
               for x in ["train", "val", "test"]}
dataset_sizes = {x: len(image_datasets[x]) for x in ["train", "val", "test"]}
class_names = image_datasets["train"].classes
num_classes = len(class_names)
os.makedirs("plots", exist_ok=True)
os.makedirs("saved_models", exist_ok=True)
print("Classes:", class_names)
print("Dataset sizes:", dataset_sizes)

# Training Configuration

In [ ]:
CONFIG = {"NUM_EPOCHS_STAGE1": 40, "NUM_EPOCHS_STAGE2": 15,
          "LEARNING_RATE": 1e-3, "LEARNING_RATE_FT": 1e-4,
          "SCHEDULER_STEP": 7, "SCHEDULER_GAMMA": 0.1}

# Training Function

In [ ]:
def train_model(model, criterion, optimizer, scheduler, dataloaders, dataset_sizes,
                num_epochs=25, device="cuda", use_amp=True):
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    since = time.time()
    best_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}"); print("-"*20)
        for phase in ["train", "val"]:
            model.train() if phase=="train" else model.eval()
            running_loss = running_corrects = 0
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase=="train"):
                    with torch.cuda.amp.autocast(enabled=use_amp):
                        outputs = model(inputs)
                        _, preds = torch.max(outputs, 1)
                        loss = criterion(outputs, labels)
                    if phase=="train":
                        scaler.scale(loss).backward()
                        scaler.step(optimizer)
                        scaler.update()
                running_loss     += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc  = running_corrects.double() / dataset_sizes[phase]
            print(f"{phase} Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.4f}")
            if phase=="train":
                scheduler.step()
                history["train_loss"].append(epoch_loss)
                history["train_acc"].append(epoch_acc.item())
            else:
                history["val_loss"].append(epoch_loss)
                history["val_acc"].append(epoch_acc.item())
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_wts = copy.deepcopy(model.state_dict())
        print()
    elapsed = time.time() - since
    print(f"Done in {elapsed//60:.0f}m {elapsed%60:.0f}s  Best val Acc: {best_acc:.4f}")
    model.load_state_dict(best_wts)
    return model, history

# Evaluation & Plotting Functions

In [ ]:
def evaluate_model(model, dataloader, device="cuda"):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            all_preds.append(model(inputs).argmax(1).cpu())
            all_labels.append(labels.cpu())
    return torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()

def print_metrics(y_true, y_pred, class_names):
    print("
=== Test Metrics ===")
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, average='macro'):.4f}")
    print(f"Recall   : {recall_score(y_true, y_pred, average='macro'):.4f}")
    print(f"F1 Score : {f1_score(y_true, y_pred, average='macro'):.4f}")
    print(classification_report(y_true, y_pred, target_names=class_names))
    cm = confusion_matrix(y_true, y_pred)
    print("Confusion Matrix:
", cm)
    return cm

def plot_confusion_matrix(cm, class_names, model_name, save_path):
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - {model_name}")
    plt.ylabel("True Label"); plt.xlabel("Predicted Label")
    plt.tight_layout(); plt.savefig(save_path, dpi=300); plt.show(); plt.close()
    print(f"Saved: {save_path}")

def plot_training_history(history, model_name, save_path):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(epochs, history["train_loss"], "b-", label="Train")
    ax1.plot(epochs, history["val_loss"],   "r-", label="Val")
    ax1.set_title(f"Loss - {model_name}"); ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.plot(epochs, history["train_acc"], "b-", label="Train")
    ax2.plot(epochs, history["val_acc"],   "r-", label="Val")
    ax2.set_title(f"Accuracy - {model_name}"); ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=300); plt.show(); plt.close()
    print(f"Saved: {save_path}")

# ResNet-50 Model Definition

In [ ]:
def create_resnet50(num_classes, pretrained=True):
    model = models.resnet50(weights="IMAGENET1K_V2" if pretrained else None)
    for param in model.parameters():
        param.requires_grad = False
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(512, num_classes),
    )
    return model

# Stage 1 — Train Classifier Head

In [ ]:
model_res = create_resnet50(num_classes).to(device)
criterion_res = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer_res = optim.AdamW(filter(lambda p: p.requires_grad, model_res.parameters()),
                             lr=CONFIG["LEARNING_RATE"], weight_decay=1e-4)
scheduler_res = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_res, T_max=CONFIG["NUM_EPOCHS_STAGE1"], eta_min=1e-6)

print("=== ResNet-50  Stage 1: Train head ===")
model_res, history_res = train_model(
    model_res, criterion_res, optimizer_res, scheduler_res,
    dataloaders, dataset_sizes,
    num_epochs=CONFIG["NUM_EPOCHS_STAGE1"], device=device)

# Stage 2 — Fine-tune Layer3 + Layer4

In [ ]:
for name, param in model_res.named_parameters():
    if "layer4" in name or "layer3" in name:
        param.requires_grad = True

optimizer_res_ft = optim.AdamW(filter(lambda p: p.requires_grad, model_res.parameters()),
                                lr=CONFIG["LEARNING_RATE_FT"], weight_decay=1e-4)
scheduler_res_ft = optim.lr_scheduler.CosineAnnealingLR(
    optimizer_res_ft, T_max=CONFIG["NUM_EPOCHS_STAGE2"], eta_min=1e-7)

print("=== ResNet-50  Stage 2: Fine-tune layer3 + layer4 ===")
model_res, history_res_ft = train_model(
    model_res, criterion_res, optimizer_res_ft, scheduler_res_ft,
    dataloaders, dataset_sizes,
    num_epochs=CONFIG["NUM_EPOCHS_STAGE2"], device=device)

for k in history_res:
    history_res[k].extend(history_res_ft[k])

torch.save(model_res.state_dict(), "saved_models/resnet50_best.pth")
print("Model saved to saved_models/resnet50_best.pth")

# Evaluation & Save Plots

In [ ]:
preds_res, labels_res = evaluate_model(model_res, dataloaders["test"], device)
cm_res = print_metrics(labels_res, preds_res, class_names)
plot_confusion_matrix(cm_res, class_names, "ResNet-50",
                      "plots/resnet50_confusion_matrix.png")
plot_training_history(history_res, "ResNet-50",
                      "plots/resnet50_training_history.png")
print("=== ResNet-50 Complete ===")

# Prediction Function

In [ ]:
def predict_image(image_path, model, class_names, device="cuda"):
    img = Image.open(image_path).convert("RGB")
    img_tensor = data_transforms["test"](img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(img_tensor), dim=1)
        conf, pred = torch.max(probs, 1)
    print(f"Prediction: {class_names[pred.item()]}  Confidence: {conf.item():.4f}")
    top5_prob, top5_idx = torch.topk(probs, min(5, len(class_names)))
    for i in range(top5_prob.shape[1]):
        print(f"  {i+1}. {class_names[top5_idx[0][i]]}: {top5_prob[0][i]:.4f}")
    return class_names[pred.item()], conf.item()